# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Schema URL:** [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- **Dataset Title:** Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd
# Define the dataset URLcroissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"
# Load the dataset metadatadataset = mlc.Dataset(croissant_url)metadata = dataset.metadata.to_json()print("Dataset Loaded:")print(f"Title: {metadata['name']}")print(f"Description: {metadata['description']}")print(f"Identifier: {metadata['identifier']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset is organized into record sets, each uniquely identified by an `@id`. Within each record set, fields (variables) are also referenced by their `@id`. Let's enumerate available record sets and their fields.

In [ ]:
# List record sets and corresponding fields/columns by @idrecord_sets = dataset.metadata.get_record_sets()print("Record Sets available (with @id):")for rs in record_sets:    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")    fields = dataset.metadata.get_fields(record_set=rs['@id'])    print("  Fields in this record set:")    for field in fields:        print(f"    - {field['@id']} (name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')})")    columns = dataset.metadata.get_columns(record_set=rs['@id'])    if columns:        print("  Columns in this record set:")        for col in columns:            print(f"    - {col['@id']} (name: {col.get('name', 'N/A')})")    print()

## 3. Data Extraction
Load data from the record sets into DataFrames for analysis. We use record set and field `@id`s (as printed above) to specify the entities.

Let's extract records for each available record set.

In [ ]:
# Extract data from each record set using their @idrecord_sets_ids = [rs['@id'] for rs in dataset.metadata.get_record_sets()]dataframes = {}
for record_set_id in record_sets_ids:    # Retrieve records using mlcroissant's record set @id    print(f"Extracting records for record set: {record_set_id}")    records = list(dataset.records(record_set=record_set_id))    if len(records) == 0:        print(f"No records found for record set: {record_set_id}")        continue    df = pd.DataFrame(records)    dataframes[record_set_id] = df    print(f"Columns for {record_set_id}: {df.columns.tolist()}")    print(df.head())    print("\n---\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Let's choose one record set as an example (first available).
- We select a numeric field using `@id` for filtering and normalization.
- We group results by a categorical field.

In [ ]:
# Select main record set and numeric/categorical fields by @idif len(dataframes) == 0:    raise ValueError("No record sets extracted.")
# Choose the first record set for demonstrationmain_record_set_id = list(dataframes.keys())[0]df = dataframes[main_record_set_id]
# Find a numeric field @id to use (e.g. Age, if present)fields = dataset.metadata.get_fields(record_set=main_record_set_id)numeric_field_id = Nonecategorical_field_id = None
for field in fields:    if field.get('dataType') in ['Integer', 'Float', 'Number'] and field['@id'] in df.columns:        numeric_field_id = field['@id']        breakfor field in fields:    if field.get('dataType') in ['Text', 'String'] and field['@id'] in df.columns:        categorical_field_id = field['@id']        break
# Check found fieldsprint(f"Using record set @id: {main_record_set_id}")print(f"Numeric field @id: {numeric_field_id}")print(f"Categorical field @id: {categorical_field_id}")
# If numeric field was not found, pick one column with numeric dtypeif numeric_field_id is None:    numerics = df.select_dtypes(include=['int64', 'float64']).columns.tolist()    if numerics:        numeric_field_id = numerics[0]        print(f"Fallback numeric field: {numeric_field_id}")    else:        raise Exception("No numeric field available for EDA.")
# Apply filter on numeric field, e.g. thresholdthreshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10filtered_df = df[df[numeric_field_id] > threshold]print(f"Filtered records with {numeric_field_id} > {threshold}:")print(filtered_df.head())
# Normalize selected numeric field for filtered recordsfiltered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()print(f"Normalized field '{numeric_field_id}' for filtered records:")print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
# Group by categorical field (if present)if categorical_field_id and categorical_field_id in filtered_df.columns:    grouped_df = filtered_df.groupby(categorical_field_id)[numeric_field_id].mean().reset_index()    print(f"Grouped (mean) data by '{categorical_field_id}':")    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset with standard Python plotting.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns
# Histogram of numeric fieldplt.figure(figsize=(7,4))sns.histplot(df[numeric_field_id], bins=10, kde=True)plt.title(f"Distribution of {numeric_field_id}")plt.xlabel(numeric_field_id)plt.show()
# If categorical field is present, boxplotif categorical_field_id and categorical_field_id in df.columns:    plt.figure(figsize=(8,5))    sns.boxplot(y=df[numeric_field_id], x=df[categorical_field_id])    plt.title(f"{numeric_field_id} by {categorical_field_id}")    plt.ylabel(numeric_field_id)    plt.xlabel(categorical_field_id)    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset by loading tabular data from Croissant schema, reviewing metadata, extracting entities using their `@id`, and performing preliminary filtering and visualization.

- We used `mlcroissant` to load and process data, referencing entities by their `@id` for reproducible, schema-driven exploration.
- Numeric attributes, such as age or biomarker counts, were filtered and normalized.
- Categorical groupings revealed possible differences across stratifications.
- Further analyses may include more domain-specific hypotheses, variable selection, and clinical outcome modeling.